# Q5. Supplier order pattern: PO frequency vs average PO value

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** How do suppliers differ in ordering frequency (PO count) and average net value per PO,
and which suppliers receive many small orders that could be consolidated into long-term contracts?

**Consolidation candidate** (simple rule): at least **5 POs** in the period **and** an average PO
value **below the median** average PO value across all suppliers. Items without a supplier are
excluded.

In [ ]:
k = w[w["Supplier"].notna()]
ps = (k.groupby(["Supplier", "supplier"])
        .agg(po_count=("Purchasing Document", "nunique"), items=("Item", "size"), net=("Net Order Value", "sum"))
        .reset_index())
ps["avg_per_po"] = ps["net"] / ps["po_count"]
MIN_POS, MED_AVG = 5, ps["avg_per_po"].median()
ps["candidate"] = (ps["po_count"] >= MIN_POS) & (ps["avg_per_po"] < MED_AVG)

print(f"Suppliers: {len(ps)} | median avg value per PO: {MED_AVG:,.0f} USD")
display(ps[["po_count", "avg_per_po", "net"]].describe().T.style.format("{:,.1f}"))
ps["po_count"].value_counts().sort_index().rename("suppliers").to_frame().T

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
rest, cand = ps[~ps["candidate"]], ps[ps["candidate"]]
jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(ps))
ax.scatter(rest["po_count"] + jitter[~ps["candidate"]], rest["avg_per_po"].clip(lower=1),
           s=30, color=SERIES[0], alpha=0.55, lw=0, label="Other suppliers")
ax.scatter(cand["po_count"] + jitter[ps["candidate"]], cand["avg_per_po"].clip(lower=1),
           s=60, color=SERIES[1], edgecolor="#fcfcfb", lw=1.5, label="Consolidation candidate")
ax.axvline(MIN_POS - 0.5, color=MUTED, lw=1, ls="--")
ax.axhline(MED_AVG, color=MUTED, lw=1, ls="--")
ax.text(ax.get_xlim()[1], MED_AVG, f"median ${MED_AVG:,.0f} / PO ", ha="right", va="bottom", color=INK_2, fontsize=9)
ax.set_yscale("log"); ax.yaxis.set_major_formatter(usd)
ax.set(title=f"PO frequency vs average PO value per supplier, {PERIOD}",
       xlabel="PO count (distinct POs)", ylabel="Average Net PO value per PO (log scale)")
ax.legend(loc="upper right")
save(fig, "q5_frequency_vs_value")

In [ ]:
cands = ps[ps["candidate"]].sort_values("po_count", ascending=False)
print(f"{len(cands)} candidates | {cands['po_count'].sum()} POs | {cands['net'].sum():,.0f} USD "
      f"({cands['net'].sum() / TOTAL:.1%} of value, {cands['po_count'].sum() / w['Purchasing Document'].nunique():.0%} of POs)")
cands[["supplier", "po_count", "items", "net", "avg_per_po"]].style.format(
    {"net": "{:,.0f}", "avg_per_po": "{:,.0f}"}).hide(axis="index")

## Findings for Antonio

**Finding.** Most suppliers are ordered from rarely: **108 of 290 received only one PO** and the median is 2 POs. The median average PO value is **$3,350**. **18 suppliers** receive ≥5 POs with a below-median average PO value. Together they account for **105 POs (14% of all POs)** but only **$106k (3.5% of value)**. Some of their average POs are below $200 (e.g. Naewkamdee Nattaya: 8 POs, $144 each; Qompany: 8 POs, $191 each).

**Decision it supports.** These 18 suppliers are candidates for consolidation. Antonio could replace the frequent small POs with a long-term contract or scheduling agreement and call-offs, cutting the processing cost per PO.

**Limitation / extra data.** The thresholds (≥5 POs, below-median value) are a simple rule of thumb. Antonio would need the internal cost per PO, and to know whether these suppliers supply the same materials, before consolidating.